<a href="https://colab.research.google.com/github/oceanicdayi/SSIF_V3/blob/main/notebooks/SSIF_V3_Colab_Tutorial_ZH_TW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SSIF_V3：Google Colab 中文實作教學

本 Notebook 依序示範：

1. 掛載 Google Drive 與 clone GitHub repository；
2. 檢查 GPU 與執行 smoke test；
3. 稽核 CWA 事件 JSON 並建立 train／validation／calibration／test；
4. 快速訓練 EW10；
5. 正式訓練 EW10–EW40；
6. 對獨立資料執行 inference；
7. 顯示主要指標、anticipatory subset 與串流 replay。

> 資料與模型保存在 Google Drive，不會上傳 GitHub。正式研究前請先閱讀 `docs/COLAB_GUIDE_ZH_TW.md`。


## 0. 選擇 GPU

Colab：`執行階段 → 變更執行階段類型 → GPU`。

In [1]:
import torch, platform
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


Python: 3.12.13
PyTorch: 2.11.0+cu128
CUDA available: True
Device: NVIDIA A100-SXM4-80GB


## 1. 掛載 Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 2. Clone SSIF_V3 並安裝套件

In [3]:
%cd /content
!rm -rf SSIF_V3
!git clone https://github.com/oceanicdayi/SSIF_V3.git
%cd /content/SSIF_V3
!git rev-parse HEAD
!python -m pip install --upgrade pip
!python -m pip install -r requirements.txt


/content
Cloning into 'SSIF_V3'...
remote: Enumerating objects: 72, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 72 (delta 20), reused 6 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (72/72), 76.73 KiB | 1.83 MiB/s, done.
Resolving deltas: 100% (20/20), done.
/content/SSIF_V3
608fe21d424a26758e45f1b2c499081cef39dbf1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 45.1 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [4]:
import pandas as pd

data_path = '/content/drive/MyDrive/00_SSIF/combined_data.csv'
df = pd.read_csv(data_path)
df.head()

,times,stids,intensity,epicenter_distance,variables,eq_info,source_file
0,"['2024-02-20T14:57:31', '2024-02-20T14:57:32',...","{'A001': {'city': '臺北市', 'town': '中正區', 'name'...","{'A001': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","{'A001': 228.951, 'A002': 238.845, 'A003': 236...","['intensity', 'epicenter_distance']","{'origin_time': '2024-02-20T14:57:31', 'longit...",/content/drive/MyDrive/00_SSIF/data/hist/20240...
1,"['2024-04-03T00:00:19', '2024-04-03T00:00:20',...","{'A001': {'city': '臺北市', 'town': '中正區', 'name'...","{'A001': [2, 2, 2, 2, 3, 2, 2, 2, 2, 2, 2, 2, ...","{'A001': 136.441, 'A002': 146.511, 'A003': 142...","['intensity', 'epicenter_distance']","{'origin_time': '2024-04-03T00:00:19', 'longit...",/content/drive/MyDrive/00_SSIF/data/hist/20240...
2,"['2024-04-02T23:58:09', '2024-04-02T23:58:10',...","{'A001': {'city': '臺北市', 'town': '中正區', 'name'...","{'A001': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","{'A001': 130.552, 'A002': 140.63, 'A003': 136....","['intensity', 'epicenter_distance']","{'origin_time': '2024-04-02T23:58:09', 'longit...",/content/drive/MyDrive/00_SSIF/data/hist/20240...
3,"['2024-04-03T00:11:26', '2024-04-03T00:11:27',...","{'A001': {'city': '臺北市', 'town': '中正區', 'name'...","{'A001': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","{'A001': 101.486, 'A002': 111.865, 'A003': 107...","['intensity', 'epicenter_distance']","{'origin_time': '2024-04-03T00:11:26', 'longit...",/content/drive/MyDrive/00_SSIF/data/hist/20240...
4,"['2024-05-06T09:45:32', '2024-05-06T09:45:33',...","{'A001': {'city': '臺北市', 'town': '中正區', 'name'...","{'A001': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","{'A001': 140.954, 'A002': 150.947, 'A003': 146...","['intensity', 'epicenter_distance']","{'origin_time': '2024-05-06T09:45:32', 'longit...",/content/drive/MyDrive/00_SSIF/data/hist/20240...


## 3. 設定研究資料路徑

請先把事件 JSON 放到 Google Drive 的 `training_archive` 與 `external_evaluation`。

In [5]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/SSIF_V3_workspace')
REPO_ROOT = Path('/content/SSIF_V3')
TRAIN_DATA = DRIVE_ROOT / 'data' / 'training_archive'
EXTERNAL_DATA = DRIVE_ROOT / 'data' / 'external_evaluation'
PREPARED_DIR = DRIVE_ROOT / 'prepared' / 'split_v1'
MODEL_DIR = DRIVE_ROOT / 'models' / 'seed_20260728'
INFERENCE_DIR = DRIVE_ROOT / 'inference' / 'external_seed_20260728'
REPLAY_DIR = DRIVE_ROOT / 'replay'
WINDOWS = [10, 15, 20, 25, 30, 35, 40]
SEED = 20260728

for path in [PREPARED_DIR, MODEL_DIR, INFERENCE_DIR, REPLAY_DIR]:
    path.mkdir(parents=True, exist_ok=True)

train_files = sorted(TRAIN_DATA.rglob('*.json'))
external_files = sorted(EXTERNAL_DATA.rglob('*.json'))
print('Training JSON:', len(train_files))
print('External JSON:', len(external_files))
print('Training examples:', train_files[:3])


Training JSON: 0
External JSON: 0
Training examples: []


## 4. Smoke test

In [ ]:
%cd /content/SSIF_V3
!python smoke_test_pipeline_v3.py


## 5. 資料稽核與四集合切分

第一次正式建立 split 時執行。資料版本改變時請建立新的 prepared 目錄。

In [ ]:
import subprocess

audit_cmd = [
    'python', 'prepare_ssif_dataset.py', 'audit-split',
    '--data-dir', str(TRAIN_DATA),
    '--output-dir', str(PREPARED_DIR),
    '--windows', *map(str, WINDOWS),
    '--label-horizon', '120',
    '--min-label-valid-fraction', '0.80',
    '--min-window-valid-fraction', '0.80',
    '--train-ratio', '0.70',
    '--validation-ratio', '0.10',
    '--calibration-ratio', '0.10',
    '--test-ratio', '0.10',
    '--split-candidates', '5000',
    '--seed', str(SEED),
]
subprocess.run(audit_cmd, cwd=REPO_ROOT, check=True)


## 6. 檢查稽核與 split

In [ ]:
import json
import pandas as pd
from IPython.display import display

with open(PREPARED_DIR / 'audit_summary.json', encoding='utf-8') as f:
    audit_summary = json.load(f)
print(json.dumps(audit_summary, ensure_ascii=False, indent=2))

file_audit = pd.read_csv(PREPARED_DIR / 'file_audit.csv')
event_audit = pd.read_csv(PREPARED_DIR / 'event_audit.csv')
event_split = pd.read_csv(PREPARED_DIR / 'event_split.csv')
display(file_audit['status'].value_counts(dropna=False).rename_axis('status').to_frame('count'))
display(event_split['split'].value_counts().rename_axis('split').to_frame('events'))
display(event_audit.head())


In [ ]:
import matplotlib.pyplot as plt

for column in ['magnitude', 'depth_km', 'max_final_class', 'positive_fraction']:
    plt.figure(figsize=(7, 4))
    for split_name, group in event_split.groupby('split'):
        plt.hist(group[column].dropna(), bins=15, alpha=0.45, label=split_name)
    plt.xlabel(column)
    plt.ylabel('Number of events')
    plt.legend()
    plt.title(f'Distribution of {column} by split')
    plt.show()


## 7. EW10 一個 epoch 快速檢查

先確認 GPU、loss、checkpoint 和 Drive 路徑正常。

In [ ]:
QUICK_MODEL_DIR = DRIVE_ROOT / 'models' / 'quick_EW10'
quick_cmd = [
    'python', 'train_ssif_v3.py', 'train-all',
    '--data-dir', str(TRAIN_DATA),
    '--split-manifest', str(PREPARED_DIR / 'split_manifest.json'),
    '--output-dir', str(QUICK_MODEL_DIR),
    '--windows', '10',
    '--label-horizon', '120',
    '--cohort', 'common',
    '--epochs', '1',
    '--batch-size', '16',
    '--eval-batch-size', '64',
    '--lr', '3e-4',
    '--seed', str(SEED),
    '--window-seed-mode', 'same',
]
if torch.cuda.is_available():
    quick_cmd.append('--amp')
subprocess.run(quick_cmd, cwd=REPO_ROOT, check=True)
print('Checkpoint:', QUICK_MODEL_DIR / 'EW10' / 'best.pt')


## 8. 正式訓練 EW10–EW40

為避免誤觸，先將 `RUN_FULL_TRAIN` 改成 `True`。正式多 seed 實驗應保留同一 split manifest。

In [ ]:
RUN_FULL_TRAIN = False

train_cmd = [
    'python', 'train_ssif_v3.py', 'train-all',
    '--data-dir', str(TRAIN_DATA),
    '--split-manifest', str(PREPARED_DIR / 'split_manifest.json'),
    '--output-dir', str(MODEL_DIR),
    '--windows', *map(str, WINDOWS),
    '--label-horizon', '120',
    '--cohort', 'common',
    '--epochs', '30',
    '--batch-size', '16',
    '--eval-batch-size', '64',
    '--lr', '3e-4',
    '--weight-decay', '1e-2',
    '--warmup-ratio', '0.10',
    '--min-precision', '0.90',
    '--seed', str(SEED),
    '--window-seed-mode', 'same',
    '--patience', '6',
    '--workers', '2',
]
if torch.cuda.is_available():
    train_cmd.append('--amp')

if RUN_FULL_TRAIN:
    subprocess.run(train_cmd, cwd=REPO_ROOT, check=True)
else:
    print('未執行正式訓練。確認路徑與參數後將 RUN_FULL_TRAIN=True。')


## 9. 顯示訓練 summary

In [ ]:
summary_path = MODEL_DIR / 'summary.json'
if summary_path.exists():
    training_summary = json.loads(summary_path.read_text(encoding='utf-8'))
    summary_df = pd.DataFrame(training_summary)
    display(summary_df[['window', 'best_epoch', 'threshold']])
else:
    print('尚未找到正式模型 summary:', summary_path)


In [ ]:
if summary_path.exists():
    rows = []
    for item in training_summary:
        alert = item['test']['alert']
        persistence = item['test']['persistence']
        rows.append({
            'window': item['window'],
            'precision': alert['precision'],
            'pod': alert['pod'],
            'f1': alert['f1'],
            'fpr': alert['fpr'],
            'persistence_precision': persistence['precision'],
            'persistence_pod': persistence['pod'],
        })
    metrics_df = pd.DataFrame(rows).sort_values('window')
    display(metrics_df)
    for metric in ['precision', 'pod', 'f1', 'fpr']:
        plt.figure(figsize=(6, 4))
        plt.plot(metrics_df['window'], metrics_df[metric], marker='o')
        plt.xlabel('Early window (s)')
        plt.ylabel(metric)
        plt.title(f'SSIF {metric} vs. early window')
        plt.grid(True, alpha=0.3)
        plt.show()


## 10. 對獨立資料執行 inference

In [ ]:
RUN_EXTERNAL_EVAL = False

eval_cmd = [
    'python', 'train_ssif_v3.py', 'evaluate-all',
    '--data-dir', str(EXTERNAL_DATA),
    '--model-root', str(MODEL_DIR),
    '--output-dir', str(INFERENCE_DIR),
    '--windows', *map(str, WINDOWS),
    '--label-horizon', '120',
    '--cohort', 'common',
    '--batch-size', '128',
    '--workers', '2',
]
if RUN_EXTERNAL_EVAL:
    subprocess.run(eval_cmd, cwd=REPO_ROOT, check=True)
else:
    print('未執行外部 inference。確認模型存在後將 RUN_EXTERNAL_EVAL=True。')


## 11. 讀取 EW20 predictions 與 anticipatory subset

In [ ]:
pred_path = INFERENCE_DIR / 'predictions_EW20.csv'
if pred_path.exists():
    pred20 = pd.read_csv(pred_path)
    display(pred20.head())
    anticipatory = pred20[(pred20['final_class'] >= 4) & (pred20['first_cross_ge4'] > 20)]
    recall = anticipatory['alert_pred'].mean() if len(anticipatory) else float('nan')
    print('EW20 rows:', len(pred20))
    print('Anticipatory positive rows:', len(anticipatory))
    print('Anticipatory recall:', recall)
else:
    print('尚未找到:', pred_path)


## 12. 事件串流 replay

In [ ]:
RUN_REPLAY = False
if RUN_REPLAY and external_files:
    example_event = external_files[0]
    replay_output = REPLAY_DIR / 'event_predictions.jsonl'
    replay_cmd = [
        'python', 'stream_ssif_v3.py', 'replay',
        '--model-root', str(MODEL_DIR),
        '--event-json', str(example_event),
        '--output', str(replay_output),
    ]
    subprocess.run(replay_cmd, cwd=REPO_ROOT, check=True)
    replay_rows = [json.loads(line) for line in replay_output.read_text(encoding='utf-8').splitlines()]
    replay_predictions = pd.DataFrame([r for r in replay_rows if r.get('type') == 'prediction'])
    display(replay_predictions.head())
else:
    print('確認模型與 external JSON 後，將 RUN_REPLAY=True。')


## 13. 保存環境與可重現性資訊

In [ ]:
import subprocess

reproducibility = {
    'git_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, text=True).strip(),
    'python': platform.python_version(),
    'torch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'cuda_device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'split_manifest': str(PREPARED_DIR / 'split_manifest.json'),
    'seed': SEED,
    'windows': WINDOWS,
}
(REPLAY_DIR / 'environment.json').write_text(
    json.dumps(reproducibility, ensure_ascii=False, indent=2), encoding='utf-8'
)
print(json.dumps(reproducibility, ensure_ascii=False, indent=2))


## 使用限制

- `internal test` 不能在反覆查看後仍稱為完全 locked test。
- external evaluation 若被用來調整模型，就不再是最終外部驗證。
- replay 是 event-aligned，不等於全天候 continuous trigger-free deployment。
- 正式論文應加入多 seed、event-cluster bootstrap、anticipatory subset、persistence baseline，以及與 eBEAR 的事件層級互補分析。
